# Quote Memory — Build Universal Index (Colab GPU)

Embeds `chunks.jsonl` with **all-MiniLM-L6-v2** (384-dim) and writes the same index layout as local `build_index.py`:

```
index/universal_v1/
  embeddings.npy
  chunks.jsonl
  meta.json
```

**Runtime:** `Runtime → Change runtime type → T4 GPU` (free tier is fine).

### Before Colab (on your Mac)
Already done if you followed the pipeline:
```bash
python scripts/ingest/parse_catalog.py
# → data/processed/universal/chunks.jsonl  (~298 MB, ~269k chunks)
```

Upload that file to Google Drive, e.g.
`My Drive/quote_search/universal/chunks.jsonl`

**Note:** This embeds the 4 catalog shows only (~269k chunks). To include **Archer**, merge after download (no re-embed):
```bash
python scripts/search/merge_indexes.py data/index/archer_full data/index/universal_v1
# → data/index/universal_with_archer/  (~335k chunks)
```

In [ ]:
!pip -q install sentence-transformers numpy

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import torch
from google.colab import drive
from sentence_transformers import SentenceTransformer

# ── edit these paths if needed ──────────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/quote_search')
CHUNKS_PATH = DRIVE_ROOT / 'universal/chunks.jsonl'
OUTPUT_DIR = DRIVE_ROOT / 'index/universal_v1'

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
RERANKER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
BATCH_SIZE = 1024  # T4: 512–1024; lower if OOM
# ─────────────────────────────────────────────────────────────────────

In [ ]:
drive.mount('/content/drive')

assert CHUNKS_PATH.is_file(), (
    f'Missing {CHUNKS_PATH}\n'
    'Upload chunks.jsonl to Drive first (see cell 1).'
)

print(f'chunks file: {CHUNKS_PATH} ({CHUNKS_PATH.stat().st_size / 1e6:.1f} MB)')
print(f'output dir:  {OUTPUT_DIR}')
print(f'torch {torch.__version__}, cuda={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu: {torch.cuda.get_device_name(0)}')

In [ ]:
print('Loading chunks.jsonl ...')
t0 = time.time()
chunks = []
with CHUNKS_PATH.open(encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))

texts = [c['text_embed'] for c in chunks]
print(f'Loaded {len(chunks):,} chunks in {time.time() - t0:.1f}s')

from collections import Counter
by_show = Counter(c.get('show_id') for c in chunks)
for show_id, n in sorted(by_show.items()):
    print(f'  {show_id}: {n:,}')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading {MODEL_NAME} on {device} ...')
model = SentenceTransformer(MODEL_NAME, device=device)

print(f'Embedding {len(texts):,} texts (batch_size={BATCH_SIZE}) ...')
t0 = time.time()
embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
    device=device,
)
embeddings = np.asarray(embeddings, dtype=np.float32)
elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s ({len(texts)/elapsed:,.0f} chunks/s)')
print(f'shape={embeddings.shape}, dtype={embeddings.dtype}')

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUTPUT_DIR / 'embeddings.npy', embeddings)

meta = {
    'model_name': MODEL_NAME,
    'reranker_model': RERANKER_MODEL,
    'chunk_count': len(chunks),
    'embedding_dim': int(embeddings.shape[1]),
}
(OUTPUT_DIR / 'meta.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')

with (OUTPUT_DIR / 'chunks.jsonl').open('w', encoding='utf-8') as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print('Saved:')
for name in ['embeddings.npy', 'chunks.jsonl', 'meta.json']:
    p = OUTPUT_DIR / name
    print(f'  {p}  ({p.stat().st_size / 1e6:.1f} MB)')
print(json.dumps(meta, indent=2))

In [ ]:
# Optional: zip for faster download to Mac
import shutil

zip_path = shutil.make_archive(str(OUTPUT_DIR), 'zip', OUTPUT_DIR)
print(f'Zip: {zip_path} ({Path(zip_path).stat().st_size / 1e6:.1f} MB)')

## After Colab

1. Download from Drive: `quote_search/index/universal_v1/` (or the zip)
2. Copy into your repo:
   ```bash
   mkdir -p data/index/universal_v1
   cp ~/Downloads/universal_v1/* data/index/universal_v1/
   ```
3. Package + upload to HuggingFace (same as Archer):
   ```bash
   bash serving/package_artifacts.sh
   ```
4. Point Railway at the new index (`SERVE_INDEX_DIR` or HF artifacts repo)

**Expected index size:** ~700 MB total (~412 MB embeddings + ~298 MB chunks.jsonl)